# SE374.R11 Bonus Extension: WSI Low-Mag to High-Mag End-to-End Clinical Pipeline

> **Dự án:** Medical WSSS - Weakly-Supervised Semantic Segmentation on Histopathology Whole Slide Images (WSI)  
> **Tiêu bản thực nghiệm:** Bệnh nhân Ung thư vú nguyên bản **TCGA-AQ-A0Y5** (`TCGA-BRCA`, Infiltrating duct carcinoma, NOS, **885.6 MB**) từ Viện Ung thư Quốc gia Hoa Kỳ (NCI GDC Portal).  
> **Công trình tham chiếu mở rộng:** *LowMagWSS (arXiv:2607.10783, 2024/2026)*  
> **Mục tiêu mở rộng (Bonus Feature):** Hiện thực hóa trọn vẹn quy trình phân vùng tự động từ góc nhìn vĩ mô độ phóng đại thấp (Low-Mag: $5\times / 10\times$) đến vi mô độ phóng đại cao (High-Mag: $40\times$) trên toàn bộ tiêu bản bệnh nhân (WSI Gigapixel), tiết kiệm $80\% - 92\%$ tài nguyên tính toán GPU.

## 1. Cài đặt môi trường & Kết nối Google Drive
Hệ thống tự động nhận diện và nạp mã nguồn theo 4 kịch bản (Zero ModuleNotFoundError):
- **Kịch bản A:** Thư mục giải nén `SE374_WSSS_Drive_Code` trong `MyDrive/SE374_WSSS_Project/`
- **Kịch bản B:** File nén `SE374_WSSS_Drive_Code.zip` trong `MyDrive/SE374_WSSS_Project/` (tự động giải nén)
- **Kịch bản C:** Mã nguồn đặt trực tiếp tại gốc `MyDrive/` (hoặc thư mục `SE374_WSSS_Drive_Code`)
- **Kịch bản D:** Chưa mount Drive hoặc Drive trống, tự động clone từ `https://github.com/khoinguyen59/medical-wsss.git`

In [ ]:
# 1. Mount Google Drive
import os, sys, zipfile, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f'[CANH BAO] Khong mount duoc Drive: {e}')

# 2. Tim kiem thu muc ma nguon SE374_WSSS_Drive_Code tren Google Drive (Kich ban A, B, C)
MYDRIVE = Path('/content/drive/MyDrive')
DRIVE_ZIPS = [
    MYDRIVE / 'SE374_WSSS_Project/SE374_WSSS_Drive_Code.zip',
    MYDRIVE / 'SE374_WSSS_Drive_Code.zip',
    Path('/content/SE374_WSSS_Drive_Code.zip'),
]
if MYDRIVE.is_dir():
    DRIVE_ZIPS += sorted(MYDRIVE.glob('*/SE374_WSSS_Drive_Code.zip')) + sorted(MYDRIVE.glob('*/*/SE374_WSSS_Drive_Code.zip'))

for zp in DRIVE_ZIPS:
    if zp.is_file() and not Path('/content/SE374_WSSS_Drive_Code/src').is_dir():
        print(f'[INFO] Dang giai nen ma nguon tu Drive: {zp}')
        with zipfile.ZipFile(zp, 'r') as zf:
            zf.extractall('/content/SE374_WSSS_Drive_Code')
        break

CANDIDATES = [
    MYDRIVE / 'SE374_WSSS_Project/SE374_WSSS_Drive_Code',
    MYDRIVE / 'SE374_WSSS_Drive_Code',
    MYDRIVE / 'SE374_WSSS_Project',
    MYDRIVE,
    Path('/content/SE374_WSSS_Drive_Code'),
    Path('/content/SE374_WSSS_Drive_Code/SE374_WSSS_Drive_Code'),
    Path('/content/medical_wsss_repo'),
    Path('/content'),
    Path.cwd(),
]
if MYDRIVE.is_dir():
    CANDIDATES += sorted(MYDRIVE.glob('*/SE374_WSSS_Drive_Code'))

CODE_ROOT = None
for p in CANDIDATES:
    if (p / 'src' / 'pipeline').is_dir() or (p / 'src' / 'colab_pkg').is_dir():
        CODE_ROOT = str(p.resolve())
        break

# Kich ban D: Neu chua co tren Drive, tu dong clone truc tiep tu GitHub
if CODE_ROOT is None:
    print('[INFO] Chua thay folder code tren Drive, dang tu dong clone tu GitHub...')
    !git clone https://github.com/khoinguyen59/medical-wsss.git /content/medical_wsss_repo
    if (Path('/content/medical_wsss_repo') / 'src' / 'pipeline').is_dir():
        CODE_ROOT = '/content/medical_wsss_repo'
    else:
        CODE_ROOT = str(Path.cwd().resolve())

if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

# 3. Cai dat OpenSlide C-library va cac thu vien WSI chuyen dung
!apt-get install -y -qq openslide-tools python3-openslide
!pip install -q openslide-python gdown albumentations tifffile
print(f'[OK] Da ket noi ma nguon thanh cong: {CODE_ROOT}')
print(f'[OK] Thu muc src: {os.path.join(CODE_ROOT, "src")}')

## 2. Nạp Thư Viện & Checkpoints Mô Hình Pretrained

In [ ]:
# Tu dong phuc hoi neu chay thang tu O 2 (Self-Healing Zero ModuleNotFoundError)
import os, sys, subprocess
from pathlib import Path
if not any((Path(p) / "src" / "colab_pkg").is_dir() for p in sys.path if p):
    for c in [Path("/content/drive/MyDrive/SE374_WSSS_Project/SE374_WSSS_Drive_Code"), Path("/content/drive/MyDrive/SE374_WSSS_Drive_Code"), Path("/content/drive/MyDrive"), Path("/content/SE374_WSSS_Drive_Code"), Path("/content/SE374_WSSS_Drive_Code/SE374_WSSS_Drive_Code"), Path("/content/medical_wsss_repo"), Path.cwd(), Path.cwd().parent]:
        if (c / "src" / "colab_pkg").is_dir():
            if str(c.resolve()) not in sys.path: sys.path.insert(0, str(c.resolve()))
            break
    if not any((Path(p) / "src" / "colab_pkg").is_dir() for p in sys.path if p):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=False)
        if (Path("/content/medical_wsss_repo") / "src" / "colab_pkg").is_dir():
            if "/content/medical_wsss_repo" not in sys.path: sys.path.insert(0, "/content/medical_wsss_repo")

import torch
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

from src.colab_pkg.downloader import ColabArtifactDownloader
from src.colab_pkg.model_loader import ColabModelLoader
from src.models.wsss_inference_methods import MultiMethodWSSSEngineV2
from src.pipeline.wsi_lowmag_highmag_pipeline import WSILowMagHighMagPipeline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[OK] Sử dụng thiết bị tính toán:', device)

# Tự động tải 2 checkpoint chính thức nếu chưa có trên ổ đĩa
downloader = ColabArtifactDownloader()
ckpt_stage1_path, ckpt_stage2_path = downloader.download_checkpoints()

# Nạp mô hình Stage-1 Net_CAM và Stage-2 DeepLabV3+
loader = ColabModelLoader(device=device)
model_stage1 = loader.load_stage1_netcam(ckpt_stage1_path)
model_stage2 = loader.load_stage2_deeplabv3plus(ckpt_stage2_path)
print('[OK] Đã nạp thành công 2 mô hình Stage-1 (Net_CAM) và Stage-2 (DeepLabV3+)!')

## 3. Khởi Tạo Pipeline WSI Low-Mag to High-Mag & Nạp Tiêu Bản Bệnh Nhân Thật (TCGA-BRCA 885.6 MB)

In [ ]:
pipeline = WSILowMagHighMagPipeline(
    patch_size=224,
    lowmag_target_size=(1024, 1024),
    tissue_threshold=0.15,
    device=device
)

# TIÊU BẢN MÔ BỆNH HỌC UNG THƯ VÚ THẬT (TCGA-BRCA) TOÀN DIỆN (885.6 MB)
# Nguồn: Viện Ung thư Quốc gia Hoa Kỳ (NCI Genomic Data Commons - GDC Portal)
# Bệnh nhân: TCGA-AQ-A0Y5 (Infiltrating duct carcinoma, NOS)
# File: TCGA-AQ-A0Y5-01Z-00-DX1.svs (Chuẩn y khoa Aperio ScanScope)
tcga_file_id = '3631967b-07b8-4275-92fb-6f66abe465ec'
tcga_filename = 'TCGA-AQ-A0Y5-01Z-00-DX1.f68f5b49-30fa-4fb6-bec6-5da9f6809d02.svs'
tcga_url = f'https://api.gdc.cancer.gov/data/{tcga_file_id}'
sample_slide_path = f'data/tcga_brca/{tcga_filename}'
os.makedirs('data/tcga_brca', exist_ok=True)

if not os.path.exists(sample_slide_path) or os.path.getsize(sample_slide_path) < 100000000:
    print(f'[INFO] Đang tải tiêu bản WSI Ung thư vú TCGA-BRCA thật từ cổng dữ liệu NCI GDC ({tcga_filename}, 885.6 MB)...')
    print('[INFO] Quá trình tải trên Google Colab mất khoảng 20-30 giây (tốc độ 30-50 MB/s)...')
    !curl -C - -L "{tcga_url}" -o "{sample_slide_path}"
    print('[OK] Đã tải tiêu bản TCGA-BRCA toàn diện thành công:', sample_slide_path, f'({os.path.getsize(sample_slide_path) / 1e6:.1f} MB)')

overview_img, orig_dims = pipeline.read_slide_overview(sample_slide_path)
print(f'[OK] Nạp tiêu bản WSI bệnh nhân TCGA-AQ-A0Y5 thành công:')
print(f' - Kích thước gốc toàn slide: {orig_dims[0]:,} x {orig_dims[1]:,} pixels')
print(f' - Kích thước tầng Low-Mag (5x/10x): {overview_img.size[0]} x {overview_img.size[1]} pixels')

## 4. Phase 1: Quét Nhanh Góc Rộng Ở Low-Mag (5x) & Bản Đồ Nhiệt Vĩ Mô

In [ ]:
import time
t_start = time.time()
tissue_mask, macro_probs = pipeline.compute_macro_tissue_heatmap(overview_img, model_stage1)
scan_sec = time.time() - t_start

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(overview_img)
axes[0].set_title('1. Tiêu bản Macro gốc (Low-Mag 5x)', fontsize=13, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(tissue_mask, cmap='gray')
axes[1].set_title('2. Mặt nạ tách mô Otsu (Loại bỏ kính trắng)', fontsize=13, fontweight='bold')
axes[1].axis('off')

im = axes[2].imshow(macro_probs[0], cmap='jet')
axes[2].set_title('3. Bản đồ nhiệt vùng nghi ngờ khối u (Macro CAM)', fontsize=13, fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

print(f'[THÀNH CÔNG] Thời gian quét toàn bộ slide ở Low-Mag: {scan_sec:.3f} giây!')

## 5. Phase 2: Đánh Giá Tỷ Lệ Cắt Giảm Tính Toán Nhờ Định Hướng Low-Mag

In [ ]:
candidates, total_possible, reduction_pct = pipeline.select_candidate_highmag_patches(
    orig_dims, tissue_mask, macro_probs
)

print('='*75)
print('BÁO CÁO HIỆU NĂNG TIẾT KIỆM TÀI NGUYÊN (LOW-MAG GUIDED REDUCTION)')
print('='*75)
print(f' - Tổng số patch lý thuyết nếu quét mù quáng cả slide 40x: {total_possible:,} patches')
print(f' - Số patch nghi ngờ được Low-Mag chọn lọc để zoom 40x:   {len(candidates):,} patches')
print(f' - TỶ LỆ CẮT GIẢM TÍNH TOÁN & THỜI GIAN GPU ĐẠT ĐƯỢC:      {reduction_pct:.2f}%')
print('='*75)

## 6. Phase 3 & 4: Zoom Cực Đại 40x (M6 SOTA) & Ghép Mặt Nạ Toàn Tiêu Bản

In [ ]:
# CHAY THAT TOAN DIEN 100% TIEU BAN TCGA-BRCA (KHONG CAT BOT SO LUONG PATCH):
# Mac dinh quet 100% toan bo moi vung mo song cua tieu ban benh nhan (max_patches = None)
RUN_FULL_SLIDE = True
max_patches = None  # Quet tron ven toan bo slide, khong cat bot bat ky patch nao!

# Chạy suy luận chi tiết trên các patch ứng viên và ghép mặt nạ
stitched_mask, report = pipeline.run_end_to_end_wsi(
    slide_input=sample_slide_path,
    model_stage2=model_stage2,
    model_stage1=model_stage1,
    method='M6_MultiScale_Pyramid_D4_Ensemble',
    slide_id='TCGA_BRCA_PATIENT_AQ_A0Y5_DX1',
    max_patches_to_infer=max_patches
)

color_stitched = pipeline.colorize_mask(stitched_mask)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(overview_img)
axes[0].set_title('Tiêu bản gốc của bệnh nhân (Macro 5x)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(color_stitched)
axes[1].set_title('Mặt nạ phân vùng tái cấu trúc toàn tiêu bản (High-Mag Stitched Mask)', fontsize=14, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f'[THÀNH CÔNG] Phân tích hoàn tất tiêu bản bệnh nhân:')
print(f' - Tỷ lệ diện tích khối u xâm lấn: {report.tumor_fraction_pct}%')
print(f' - Tổng thời gian quét & phân đoạn: {report.total_elapsed_time_sec} giây')

## 7. Bảng Tóm Tắt Chẩn Đoán Lâm Sàng Cho Bác Sĩ (Clinical Summary)

In [ ]:
print('='*70)
print(f'PHIẾU KẾT QUẢ PHÂN ĐOẠN GIẢI PHẪU BỆNH SỐ (WSI AI REPORT): {report.slide_id}')
print('='*70)
print(f'Phương pháp suy luận:          {report.selected_method}')
print(f'Độ phân giải phân tích:        {report.macro_dimensions[0]}x{report.macro_dimensions[1]} pixels')
print(f'Diện tích mô sống phát hiện:   {report.tissue_area_px:,} pixels')
print(f'Diện tích Khối u (Tumor):      {report.tumor_area_px:,} pixels ({report.tumor_fraction_pct}%)')
print(f'Diện tích Mô đệm (Stroma):     {report.stroma_area_px:,} pixels')
print(f'Diện tích Bạch cầu (Lympho):   {report.lymphocyte_area_px:,} pixels')
print(f'Diện tích Hoại tử (Necrosis):  {report.necrosis_area_px:,} pixels')
print(f'Thời gian quét Low-Mag:        {report.lowmag_scan_time_sec} giây')
print(f'Thời gian suy luận High-Mag:   {report.highmag_inference_time_sec} giây')
print(f'Hiệu quả giảm tải tính toán:   TIẾT KIỆM {report.compute_reduction_pct}%')
print('='*70)